# Step 2 - Find clusters of closely related compounds

**What this notebook does.** It groups compounds into clusters of closely related molecules in two
representations - MiniMol (chemical structure, Step 0.2) and Chemical Checker sign3 (bioactivity,
Step 0.1) - and saves each compound's cluster label.

**Why.** Step 3 uses these clusters to ask a focused question: for compounds that clearly belong to a
well-defined chemical (or bioactivity) family, do the image-based representations recover the same
neighbours?

**Methods**
* **UMAP** squeezes the high-dimensional embeddings into two dimensions, for visualization only
  (cosine distance, 15 neighbours).
* **HDBSCAN** is a density-based clustering method. It finds groups of points packed more densely than
  their surroundings and labels points that belong to no dense group as *noise* (cluster −1); the number
  of clusters does not have to be chosen in advance. The settings (minimum cluster size 15, minimum samples
  30, "leaf" cluster selection) favour small, tight clusters.
* **Hierarchical-clustering heatmap** - a visual check that the clusters are coherent.

**Inputs.** `minimol_embeddings__Kim_et_al_subset_final.parquet` (Step 0.2),
`JUMPCP_Sign3_CC_final.parquet` (Step 0.1).

**Outputs.** `minimol_clusters.csv` and `sign3_clusters.csv`. In Step 3, 749 MiniMol compounds and
1,436 sign3 compounds fell in clusters (the rest were noise).

## Setup

All libraries used in this notebook are imported in the next cell. Locally they are installed by `uv sync`
(see the main README); on Google Colab, first run
`!pip install git+https://github.com/axiomcura/ModelBenchmarking.git`, which installs the same dependencies.

**Hardware.** Requires an **NVIDIA GPU with RAPIDS cuML** (GPU versions of UMAP, HDBSCAN and PCA). cuML is not part
of `pyproject.toml` because it is only available on Linux with NVIDIA GPUs; on a Colab GPU runtime install it with
`!pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com`, or see this folder's README.

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from cuml.cluster import HDBSCAN as cuHDBSCAN
from cuml.decomposition import PCA as cuPCA
from cuml.manifold import UMAP as cuUMAP
from IPython.display import display
from sklearn.preprocessing import normalize

## Part A - MiniMol (chemical structure)

### A.1 Load the embeddings
Keeps the numeric embedding columns (text/identifier columns are excluded) and drops rows with missing values.

In [2]:
print("📥 Loading Minimol Parquet from Google Cloud Storage...")

gcs_input_path = "gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_embeddings__Kim_et_al_subset_final.parquet"
gcs_output_csv = "gs://calico-reference-embeddings/Annotations_from_Roman/Bayer_data_harmonized/minimol_embeddings/minimol_clusters.csv"

# Load the minimol embeddings from the Parquet file in GCS
df = pd.read_parquet(gcs_input_path)

print("⚙️ Isolating numerical embeddings for math operations...")

# List all possible metadata/text columns we want to ignore
metadata_cols = [
    "SMILES",
    "SMILES_standard",
    "smiles",
    "CID",
    "PubChem_CID",
    "InChIKey",
    "Metadata_InChIKey",
    "Metadata_JCP2022",
    "perturbation_ID",
    "plate",
    "well",
    "Batch",
    "Metadata_broad_sample",
]

# Keep only the actual embedding columns
emb_cols = [c for c in df.columns if c not in metadata_cols]

# Safely extract ONLY the floating-point columns
df_numeric = df[emb_cols].select_dtypes(include=[np.number]).copy()
df_numeric = df_numeric.dropna()

# 🚨 Explicitly save the column names so we can drop them from the final metadata table!
dim_cols = df_numeric.columns.tolist()

X = df_numeric.values
num_molecules = len(X)
num_dimensions = X.shape[1]

print(f"   ✅ Using ALL {num_molecules:,} valid molecules and {num_dimensions:,} dimensions.")

if num_molecules == 0:
    raise ValueError("DataFrame is completely empty! Check the data loading step.")

# Ensure we have an InChIKey column for the final export
if "InChIKey" not in df.columns and df.index.name != "InChIKey":
    # Attempt auto-detection
    inchikey_col = next((c for c in df.columns if "inchikey" in c.lower()), None)
    if inchikey_col:
        df = df.rename(columns={inchikey_col: "InChIKey"})
    else:
        print(
            "⚠️ Warning: No 'InChIKey' column detected in the original dataframe. It will be missing from the output CSV."
        )

### A.2 UMAP map of chemical space
UMAP squeezes the embeddings into two dimensions for visualization only.

In [3]:
print("\n🌌 Running GPU UMAP on ALL molecules...")
start_umap = time.time()

# cuUMAP runs natively on the GPU. output_type='numpy' allows Matplotlib to plot it!
reducer = cuUMAP(n_components=2, metric="cosine", n_neighbors=15, min_dist=0.1, random_state=42, output_type="numpy")
embedding_2d = reducer.fit_transform(X)

print(f"   -> GPU UMAP completed in {time.time() - start_umap:.2f} seconds.")

plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], alpha=0.5, s=10, color="#2C7BB6")  # Blue for minimol
plt.title(f"1. UMAP Projection of Minimol chemical space (N={num_molecules:,})", fontsize=16, pad=15)
plt.xlabel("UMAP 1", fontsize=12)
plt.ylabel("UMAP 2", fontsize=12)
plt.grid(False)
sns.despine()
plt.tight_layout()
plt.show()

### A.3 HDBSCAN clustering
Vectors are scaled to unit length (so Euclidean distance behaves like cosine distance) and clustered in the full-dimensional space; compounds in no dense group are labelled noise (−1).

In [4]:
print("\n🚀 Running GPU HDBSCAN on FULL High-Dimensional Space...")
start_hdbscan = time.time()

# L2 Normalize vectors so Euclidean distance acts exactly like Cosine distance
X_norm = normalize(X, norm="l2")

# Tuned for tight clusters
clusterer = cuHDBSCAN(
    min_cluster_size=15, min_samples=30, metric="euclidean", cluster_selection_method="leaf", output_type="numpy"
)
cluster_labels = clusterer.fit_predict(X_norm)

# Count the number of distinct clusters and noise points
num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
num_noise = np.sum(cluster_labels == -1)

print(f"   -> GPU HDBSCAN completed in {time.time() - start_hdbscan:.2f} seconds.")
print(f"   -> Found {num_clusters} distinct clusters.")
print(f"   -> {num_noise:,} molecules ({(num_noise / num_molecules) * 100:.1f}%) classified as background noise.")

# Plot UMAP colored by GPU HDBSCAN Clusters
plt.figure(figsize=(10, 8))
noise_mask = cluster_labels == -1
plt.scatter(
    embedding_2d[noise_mask, 0], embedding_2d[noise_mask, 1], color="lightgray", alpha=0.3, s=10, label="Noise (-1)"
)

cluster_mask = cluster_labels != -1
plt.scatter(
    embedding_2d[cluster_mask, 0],
    embedding_2d[cluster_mask, 1],
    c=cluster_labels[cluster_mask],
    cmap="Spectral",
    alpha=0.7,
    s=15,
)

plt.title(f"2. GPU HDBSCAN Clusters (n={num_clusters}) mapped on chemical space UMAP", fontsize=16, pad=15)
plt.xlabel("UMAP 1", fontsize=12)
plt.ylabel("UMAP 2", fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

### A.4 Hierarchical-clustering heatmap
Clustered compounds on 20 principal components; rows are coloured by HDBSCAN cluster, as a visual check that clusters are coherent.

In [5]:
print("\n🌳 Running GPU PCA and Hierarchical Clustering on valid HDBSCAN groups...")

# Filter out noise points for PCA and Hierarchical Clustering
X_clustered_only = X[cluster_mask]
labels_clustered_only = cluster_labels[cluster_mask]
num_valid_mols = len(X_clustered_only)

print(f"   -> Stripped noise. Processing {num_valid_mols:,} clustered molecules.")

if num_valid_mols >= 2:
    # PCA without random_state for cuML compatibility
    print("   -> Reducing dimensionality to 20 Principal Components on GPU...")
    pca = cuPCA(n_components=20, output_type="numpy")
    X_pca = pca.fit_transform(X_clustered_only)

    variance_kept = np.sum(pca.explained_variance_ratio_) * 100
    print(f"   -> PCA complete. Preserved {variance_kept:.1f}% of variance.")

    df_hc_pca = pd.DataFrame(X_pca, columns=[f"PC {i + 1}" for i in range(20)])

    # Map the HDBSCAN cluster IDs to their respective colors for the rows
    unique_labels = np.unique(labels_clustered_only)
    palette = sns.color_palette("Spectral", len(unique_labels))
    lut = dict(zip(unique_labels, palette))
    row_colors = pd.Series(labels_clustered_only).map(lut).values

    # Calculate expected RAM usage for SciPy
    expected_ram_gb = (num_valid_mols * (num_valid_mols - 1) / 2 * 8) / (1024**3)
    print(f"   -> Expected RAM allocation for SciPy distance matrix: ~{expected_ram_gb:.2f} GB")
    print("   -> Calculating linkage and drawing Clustermap...")

    g = sns.clustermap(
        df_hc_pca,
        metric="cosine",
        method="average",
        cmap="coolwarm",
        figsize=(12, 10),
        standard_scale=1,
        row_cluster=True,
        col_cluster=False,
        row_colors=row_colors,
        yticklabels=False,
        xticklabels=True,
    )

    g.fig.suptitle(f"3. Hierarchical Clustering (ALL {num_valid_mols:,} Clustered Mols, 20-D PCA)", y=1.02, fontsize=16)
    g.ax_heatmap.set_xlabel(f"Principal Components (Preserved {variance_kept:.1f}% Variance)")
    g.ax_heatmap.set_ylabel("Molecules (Color-coded by HDBSCAN Cluster)")
    plt.show()

### A.5 Save the cluster table
One row per compound with its cluster ID, UMAP coordinates and metadata (no embedding columns); clustered compounds first.

In [6]:
print("\n📋 Assembling the final Metadata + Cluster Membership table...")

# We use df.loc to ensure we only grab rows that survived the dropna() step
df_results = df.loc[df_numeric.index].copy()

# Insert the cluster labels and UMAP coordinates
df_results.insert(0, "HDBSCAN_Cluster_ID", cluster_labels)
df_results.insert(1, "UMAP_X", embedding_2d[:, 0])
df_results.insert(2, "UMAP_Y", embedding_2d[:, 1])

# Make sure the index is explicitly preserved as a column
if df_results.index.name:
    df_results = df_results.reset_index()

# 🚨 THE FIX: Ensure InChIKey is explicitly included in the final output
# If InChIKey isn't present in the resulting dataframe, this will alert the user.
if "InChIKey" not in df_results.columns:
    print("\n⚠️ WARNING: 'InChIKey' column was not found during export! The CSV will lack InChIKeys.")
else:
    print("   -> InChIKey column successfully verified for export.")

# Drop all the raw dimensions so the CSV output is extremely clean
df_metadata_only = df_results.drop(columns=dim_cols, errors="ignore")

# Sort so clustered molecules are at the top, grouped by Cluster_ID, and noise is at the bottom
df_metadata_only["is_noise"] = df_metadata_only["HDBSCAN_Cluster_ID"] == -1
df_metadata_only = df_metadata_only.sort_values(by=["is_noise", "HDBSCAN_Cluster_ID"]).drop(columns=["is_noise"])

print("💾 Saving the final table to Google Cloud Storage...")
df_metadata_only.to_csv(gcs_output_csv, index=False)

print("🎉 SUCCESS! File saved to:")
print(f"   {gcs_output_csv}")

print("\n" + "=" * 80)
print("🏆 FINAL CLUSTER MEMBERSHIP TABLE (Preview)")
print("=" * 80)
display(df_metadata_only.head(15))

## Part B - Chemical Checker sign3 (bioactivity)
The same procedure on the 3,200-dimensional sign3 signatures. (Some printed messages still say "Minimol"; the data loaded is sign3.)

### B.1 Load the embeddings
Keeps the numeric embedding columns (text/identifier columns are excluded) and drops rows with missing values.

In [7]:
print("📥 Loading Minimol Parquet from Google Cloud Storage...")

gcs_input_path = (
    "gs://calico-reference-embeddings/embeddings/Chemical_Checker_Embeddings/parquets/JUMPCP_Sign3_CC_final.parquet"
)
gcs_output_csv = "gs://calico-reference-embeddings/Chemical_Checker_Embeddings/parquets/sign3_clusters.csv"

df = pd.read_parquet(gcs_input_path)

print("⚙️ Isolating numerical embeddings for math operations...")

# List all possible metadata/text columns we want to ignore
metadata_cols = [
    "SMILES",
    "SMILES_standard",
    "smiles",
    "CID",
    "PubChem_CID",
    "InChIKey",
    "Metadata_InChIKey",
    "Metadata_JCP2022",
    "perturbation_ID",
    "plate",
    "well",
    "Batch",
    "Metadata_broad_sample",
]

# Keep only the actual embedding columns
emb_cols = [c for c in df.columns if c not in metadata_cols]

# Safely extract ONLY the floating-point columns
df_numeric = df[emb_cols].select_dtypes(include=[np.number]).copy()
df_numeric = df_numeric.dropna()

# 🚨 Explicitly save the column names so we can drop them from the final metadata table!
dim_cols = df_numeric.columns.tolist()

X = df_numeric.values
num_molecules = len(X)
num_dimensions = X.shape[1]

print(f"   ✅ Using ALL {num_molecules:,} valid molecules and {num_dimensions:,} dimensions.")

if num_molecules == 0:
    raise ValueError("DataFrame is completely empty! Check the data loading step.")

# Ensure we have an InChIKey column for the final export
if "InChIKey" not in df.columns and df.index.name != "InChIKey":
    # Attempt auto-detection
    inchikey_col = next((c for c in df.columns if "inchikey" in c.lower()), None)
    if inchikey_col:
        df = df.rename(columns={inchikey_col: "InChIKey"})
    else:
        print(
            "⚠️ Warning: No 'InChIKey' column detected in the original dataframe. It will be missing from the output CSV."
        )

### B.2 UMAP map of chemical space
UMAP squeezes the embeddings into two dimensions for visualization only.

In [8]:
print("\n🌌 Running GPU UMAP on ALL molecules...")
start_umap = time.time()

# cuUMAP runs natively on the GPU. output_type='numpy' allows Matplotlib to plot it!
reducer = cuUMAP(n_components=2, metric="cosine", n_neighbors=15, min_dist=0.1, random_state=42, output_type="numpy")
embedding_2d = reducer.fit_transform(X)

print(f"   -> GPU UMAP completed in {time.time() - start_umap:.2f} seconds.")

plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], alpha=0.5, s=10, color="#2C7BB6")  # Blue for minimol
plt.title(f"1. UMAP Projection of ChemChecker (Sign3) chemical space (N={num_molecules:,})", fontsize=16, pad=15)
plt.xlabel("UMAP 1", fontsize=12)
plt.ylabel("UMAP 2", fontsize=12)
plt.grid(False)
sns.despine()
plt.tight_layout()
plt.show()

### B.3 HDBSCAN clustering
Vectors are scaled to unit length (so Euclidean distance behaves like cosine distance) and clustered in the full-dimensional space; compounds in no dense group are labelled noise (−1).

In [9]:
print("\n🚀 Running GPU HDBSCAN on FULL High-Dimensional Space...")
start_hdbscan = time.time()

# L2 Normalize vectors so Euclidean distance acts exactly like Cosine distance
X_norm = normalize(X, norm="l2")

# Tuned for tight clusters
clusterer = cuHDBSCAN(
    min_cluster_size=15, min_samples=30, metric="euclidean", cluster_selection_method="leaf", output_type="numpy"
)
cluster_labels = clusterer.fit_predict(X_norm)

num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
num_noise = np.sum(cluster_labels == -1)

print(f"   -> GPU HDBSCAN completed in {time.time() - start_hdbscan:.2f} seconds.")
print(f"   -> Found {num_clusters} distinct clusters.")
print(f"   -> {num_noise:,} molecules ({(num_noise / num_molecules) * 100:.1f}%) classified as background noise.")

# Plot UMAP colored by GPU HDBSCAN Clusters
plt.figure(figsize=(10, 8))
noise_mask = cluster_labels == -1
plt.scatter(
    embedding_2d[noise_mask, 0], embedding_2d[noise_mask, 1], color="lightgray", alpha=0.3, s=10, label="Noise (-1)"
)

cluster_mask = cluster_labels != -1
plt.scatter(
    embedding_2d[cluster_mask, 0],
    embedding_2d[cluster_mask, 1],
    c=cluster_labels[cluster_mask],
    cmap="Spectral",
    alpha=0.7,
    s=15,
)

plt.title(f"2. GPU HDBSCAN Clusters (n={num_clusters}) mapped on chemical space UMAP", fontsize=16, pad=15)
plt.xlabel("UMAP 1", fontsize=12)
plt.ylabel("UMAP 2", fontsize=12)
sns.despine()
plt.tight_layout()
plt.show()

### B.4 Hierarchical-clustering heatmap
Clustered compounds on 20 principal components; rows are coloured by HDBSCAN cluster, as a visual check that clusters are coherent.

In [10]:
print("\n🌳 Running GPU PCA and Hierarchical Clustering on valid HDBSCAN groups...")

X_clustered_only = X[cluster_mask]
labels_clustered_only = cluster_labels[cluster_mask]
num_valid_mols = len(X_clustered_only)

print(f"   -> Stripped noise. Processing {num_valid_mols:,} clustered molecules.")

if num_valid_mols >= 2:
    # PCA without random_state for cuML compatibility
    print("   -> Reducing dimensionality to 20 Principal Components on GPU...")
    pca = cuPCA(n_components=20, output_type="numpy")
    X_pca = pca.fit_transform(X_clustered_only)

    variance_kept = np.sum(pca.explained_variance_ratio_) * 100
    print(f"   -> PCA complete. Preserved {variance_kept:.1f}% of variance.")

    df_hc_pca = pd.DataFrame(X_pca, columns=[f"PC {i + 1}" for i in range(20)])

    # Map the HDBSCAN cluster IDs to their respective colors for the rows
    unique_labels = np.unique(labels_clustered_only)
    palette = sns.color_palette("Spectral", len(unique_labels))
    lut = dict(zip(unique_labels, palette))
    row_colors = pd.Series(labels_clustered_only).map(lut).values

    # Calculate expected RAM usage for SciPy
    expected_ram_gb = (num_valid_mols * (num_valid_mols - 1) / 2 * 8) / (1024**3)
    print(f"   -> Expected RAM allocation for SciPy distance matrix: ~{expected_ram_gb:.2f} GB")
    print("   -> Calculating linkage and drawing Clustermap...")

    g = sns.clustermap(
        df_hc_pca,
        metric="cosine",
        method="average",
        cmap="coolwarm",
        figsize=(12, 10),
        standard_scale=1,
        row_cluster=True,
        col_cluster=False,
        row_colors=row_colors,
        yticklabels=False,
        xticklabels=True,
    )

    g.fig.suptitle(f"3. Hierarchical Clustering (ALL {num_valid_mols:,} Clustered Mols, 20-D PCA)", y=1.02, fontsize=16)
    g.ax_heatmap.set_xlabel(f"Principal Components (Preserved {variance_kept:.1f}% Variance)")
    g.ax_heatmap.set_ylabel("Molecules (Color-coded by HDBSCAN Cluster)")
    plt.show()

### B.5 Save the cluster table
One row per compound with its cluster ID, UMAP coordinates and metadata (no embedding columns); clustered compounds first.

In [11]:
print("\n📋 Assembling the final Metadata + Cluster Membership table...")

# We use df.loc to ensure we only grab rows that survived the dropna() step
df_results = df.loc[df_numeric.index].copy()

# Insert the cluster labels and UMAP coordinates
df_results.insert(0, "HDBSCAN_Cluster_ID", cluster_labels)
df_results.insert(1, "UMAP_X", embedding_2d[:, 0])
df_results.insert(2, "UMAP_Y", embedding_2d[:, 1])

# Make sure the index is explicitly preserved as a column
if df_results.index.name:
    df_results = df_results.reset_index()

# 🚨 THE FIX: Ensure InChIKey is explicitly included in the final output
# If InChIKey isn't present in the resulting dataframe, this will alert the user.
if "InChIKey" not in df_results.columns:
    print("\n⚠️ WARNING: 'InChIKey' column was not found during export! The CSV will lack InChIKeys.")
else:
    print("   -> InChIKey column successfully verified for export.")

# Drop all the raw dimensions so the CSV output is extremely clean
df_metadata_only = df_results.drop(columns=dim_cols, errors="ignore")

# Sort so clustered molecules are at the top, grouped by Cluster_ID, and noise is at the bottom
df_metadata_only["is_noise"] = df_metadata_only["HDBSCAN_Cluster_ID"] == -1
df_metadata_only = df_metadata_only.sort_values(by=["is_noise", "HDBSCAN_Cluster_ID"]).drop(columns=["is_noise"])

print("💾 Saving the final table to Google Cloud Storage...")
df_metadata_only.to_csv(gcs_output_csv, index=False)

print("🎉 SUCCESS! File saved to:")
print(f"   {gcs_output_csv}")

print("\n" + "=" * 80)
print("🏆 FINAL CLUSTER MEMBERSHIP TABLE (Preview)")
print("=" * 80)
display(df_metadata_only.head(15))